# Do secondary controls add anything beyond glacier area and aridity?

Fig. 2 attributes the latitudinal pattern in glacier melt contribution to relative glacier
area modulated by aridity. This notebook tests whether any *other* catchment attribute adds
explanatory power once those two are accounted for.

The test is **cross-validated skill**, not residual correlation. A residual-correlation screen
was tried first and rejected: its ranking flipped completely depending on how the primary
model was specified (raw-linear vs log vs random forest shared almost no top attributes),
and because a linear fit does not fully absorb aridity, the top hits were simply whichever
attributes proxy aridity best (low-precip. frequency has rho = -0.77 with aridity). With
n = 652, p < 0.05 is reached at |rho| > 0.08, so significance testing flagged noise as real.

Asking "how much predictive skill does this attribute *add*" avoids all three problems: it
is independent of the primary model's functional form, it cannot be gamed by collinearity
with aridity (a redundant attribute adds nothing), and out-of-sample scoring means noise
scores at or below zero instead of appearing significant.

In [ ]:
# general
import pandas as pd

# spatial
import geopandas as gpd
import xarray as xr

# stats
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict, cross_val_score

# plotting
from sklearn.preprocessing import MinMaxScaler
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils import load_config
from src.plotting import (
    base_layout, add_panel_label, save_figure,
    MARKER_LINE_COLOR, MUTED_TEXT_COLOR,
)
cfg = load_config()

## Data

In [ ]:
common_period = slice("2001-01-01", "2024-12-31")

# polygons / shapes
hydroatlas_shape = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")

# total runoff in mm/day
hydroatlas_runoff = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc", chunks = "auto").q_mm_sim
hydroatlas_runoff = hydroatlas_runoff.sel(date=common_period)

# glacier melt in m3/s -> mm/day
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc", chunks = "auto").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.reindex(gauge_id=hydroatlas_shape.index, fill_value=0)
hydroatlas_gmelt = hydroatlas_gmelt.sel(date=common_period)
hydroatlas_gmelt = (hydroatlas_gmelt * 1000 * 86400) / (hydroatlas_shape.basin_area.values * 1e6)[:, None]

# annual ratio 
ratio = xr.merge([hydroatlas_runoff, hydroatlas_gmelt], join="outer")
ratio = ratio.resample(date="YS").sum().mean(dim = "date")
ratio = (ratio.glacier_melt * 100 / ratio.q_mm_sim).to_series()
hydroatlas_shape = hydroatlas_shape.join(ratio.rename("runoff_gmelt_ratio"))
hydroatlas_shape = hydroatlas_shape[hydroatlas_shape.glacier_area_RGI60 > 0]

## Cross-validated variance explained

Candidates are the physical HydroATLAS attributes, excluding the two primary controls, the
components aridity is built from (`ev_mean`, `prcp_mean`), routing/topology IDs, and
near-duplicate columns (median/std elevation, alternative glacier volumes, sub/upstream area).

In [ ]:
primary = ["glacier_area_RGI60", "aridity_ERA5"]
candidates = [
    "slope_mean", "elev_mean", "prcp_pci_ERA5", "frac_snow_ERA5", "solid_prcp_ERA5",
    "basin_area", "low_prec_freq_ERA5", "high_prec_freq_ERA5", "forest_cover",
    "shrubs_veg_cover", "basin_cenlat", "aspect_mean", "gc_index",
]

model_data = hydroatlas_shape.dropna(subset=["runoff_gmelt_ratio"] + primary + candidates)
target = model_data["runoff_gmelt_ratio"]
kfold = KFold(5, shuffle=True, random_state=0)

def cv_r2(predictors):
    """Out-of-sample R2 of a random forest on `predictors` (a DataFrame).

    A forest rather than a linear fit so the glacier-area x aridity interaction and any
    curvature are captured without hand-specifying a functional form -- the point is to
    bound what the primary controls can explain, not to interpret coefficients.
    """
    model = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=0, n_jobs=-1)
    return cross_val_score(model, predictors, target, cv=kfold, scoring="r2").mean()

ladder = pd.Series({
    "Glacier area alone":        cv_r2(model_data[["glacier_area_RGI60"]]),
    "Aridity alone":             cv_r2(model_data[["aridity_ERA5"]]),
    "Glacier area + aridity":    cv_r2(model_data[primary]),
    "All attributes":       cv_r2(model_data[primary + candidates]),
})
print(ladder.round(3).to_string())
print(f"\nn = {len(model_data)} basins with glacier cover > 0")

In [ ]:
baseline = ladder["Glacier area + aridity"]

# Per-basin out-of-sample predictions from each model. cross_val_predict reuses the folds
# from the scores above, so every basin is predicted by a model that never saw it and the
# two prediction sets are comparable basin by basin.
def cv_predict(predictors):
    model = RandomForestRegressor(n_estimators=150, min_samples_leaf=5, random_state=0, n_jobs=-1)
    return cross_val_predict(model, predictors, target, cv=kfold)

pred_primary = pd.Series(cv_predict(model_data[primary]), index=model_data.index)
pred_full = pd.Series(cv_predict(model_data[primary + candidates]), index=model_data.index)

change = (pred_full - pred_primary).abs()

## Plot

In [ ]:
# aridity colorscale and basin-area marker sizing, shared with fig_runoff_glacier_area
colorscale = ["#f3e79b", "#eb7f86", "#ce6693", "#a059a0", "#5c53a5"]
scaler = MinMaxScaler(feature_range=(10, 30))
PRIMARY_COLOR = "#31688e"

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.10, row_heights=[0.62, 0.38])

# Panel a: the same basins predicted by both models. Points sitting on the 1:1 line mean the
axis_max = max(pred_primary.max(), pred_full.max()) * 1.09

fig.add_trace(
    go.Scatter(
        x=[-5, axis_max], y=[-5, axis_max],
        mode="lines",
        line=dict(color="grey", width=1, dash="dot"),
        showlegend=False,
    ),
    row=1, col=1,
)

fig.add_trace(
    go.Scatter(
        x=pred_primary, y=pred_full,
        mode="markers",
        marker=dict(
            size=scaler.fit_transform(model_data[["basin_area"]]).flatten(),
            color=model_data.aridity_ERA5,
            colorscale=colorscale,
            opacity=0.9,
            colorbar=dict(
                title="Aridity<br>index<br>(PET/P)<br> ",
                len=0.55, y=0.735, yanchor="middle",
                thickness=20,
                ticks="outside",
                tickvals=[0, 0.2, 0.4, 0.6, 0.8, 1],
                ticktext=["0", "0.2", "0.4", "0.6", "0.8", ">1"],
            ),
            line=dict(width=0.5, color=MARKER_LINE_COLOR),
            cmin=0, cmax=1,
            showscale=True,
        ),
        showlegend=False,
    ),
    row=1, col=1,
)

# Legend for point size (basin area). Passed as a frame so the scaler sees the same column
# name it was fitted on.
size_legend_values = [100, 1000, 10000]
size_legend_sizes = scaler.transform(
    pd.DataFrame({"basin_area": size_legend_values})
).flatten()

for value, size in zip(size_legend_values, size_legend_sizes):
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None],
            mode="markers",
            marker=dict(size=size, color=MARKER_LINE_COLOR, line=dict(width=0.5, color=MARKER_LINE_COLOR)),
            legendgroup="size",
            name=f"{value:,.0f}",
            showlegend=True,
        ),
        row=1, col=1,
    )

fig.add_annotation(
    xref="paper", yref="paper", x=0.40, y=1.02,
    xanchor="right", yanchor="middle",
    text="Basin area (km²):", showarrow=False, font=dict(size=14),
)

# Panel b: what each primary control is worth alone and together
gain = ladder["All attributes"] - baseline
ladder_text = [f"{v:.2f}" for v in ladder.values]

fig.add_trace(
    go.Bar(
        x=ladder.values,
        y=[name.replace(" area", " area<br>") for name in ladder.index],
        orientation="h",
        marker=dict(color=PRIMARY_COLOR, line=dict(width=0.5, color=MARKER_LINE_COLOR)),
        text=ladder_text,
        textposition="outside",
        textfont=dict(size=13),
        cliponaxis=False,
        showlegend=False,
    ),
    row=2, col=1,
)

fig.update_xaxes(title="Predicted contribution from glacier area and aridity (%)", title_standoff=5,
                 range=[-2, axis_max], row=1, col=1)
fig.update_yaxes(title="Predicted contribution from all predictors (%)", title_standoff=5,
                 range=[-2, axis_max], row=1, col=1)

fig.update_xaxes(title="Cross-validated R²", title_standoff=5, range=[0, 1.02], row=2, col=1)
fig.update_yaxes(autorange="reversed", row=2, col=1)

add_panel_label(fig, "a)", x=0.96, y=0.995)
add_panel_label(fig, "b)", x=0.96, y=0.335)

fig.update_layout(**base_layout(
    width=660, height=800,
    legend=dict(orientation="h", y=1.04, x=0.40, bgcolor="rgba(255,255,255,0)",
                traceorder="normal", tracegroupgap=0),
    margin=dict(l=100, r=30, b=55, t=40),
))

save_figure(fig, "figures/fig_runoff_secondary_controls.svg", width=660, height=800)
fig.show()

**Figure X. Glacier area and aridity capture most of the predictive skill for glacier melt contribution relative to total runoff. (a) Mean annual glacier melt contribution to runoff for all glacierized basins, predicted by a random forest using glacier area and aridity only against one using all HydroATLAS catchment attributes. All predictions are out-of-sample (5-fold cross-validation), and the dotted line is 1:1. Points are colored by aridity index (PET/P) and scaled by basin area. (b) Cross-validated R² for each predictor set.